In [11]:
# ============================================================
# CELL 1 — Imports & path setup
# ============================================================
import sys, os, time, logging

LIB_PATH = "/home/dhanesh/dhanesh/TMYTEK/TLKCore_v2.4.4_Linux_Python3.12-x86_64/lib/"
if LIB_PATH not in sys.path:
    sys.path.insert(0, LIB_PATH)

from tlkcore import TLKCoreService, DevInterface, RetCode, RFMode, UDState, BeamType

logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("System")

# ── Known device SNs from your scan ─────────────────────────
SN_UC  = "UD-BS25020007-24"   # Up-converter   (UDBox)
SN_DC  = "UD-BS25020006-24"   # Down-converter (UDBox)
SN_TX  = "D2426E012-28"       # TX Beamformer  (BBox)
SN_RX  = "D2426E015-28"       # RX Beamformer  (BBox)

# ============================================================

In [12]:
# CELL 2 — Start service & scan
# ============================================================
service = TLKCoreService()
logger.info("TLKCoreService v%s | running: %s" % (service.version, service.running))

service.scanDevices(interface=DevInterface.LAN)
scan_dict = service.getScanInfo().RetData

logger.info("── Scan results ──────────────────────────")
for sn, (addr, devtype, in_dfu) in scan_dict.items():
    logger.info("  SN: %-25s | IP: %-18s | Type: %d" % (sn, addr, devtype))


# ============================================================

2026-03-18 16:13:27.619 - TLKCoreService - INFO : ========= TLKService v2.4.4 - local =========
2026-03-18 16:13:27.619 - TLKCoreService - INFO : Working Root directory: /home/dhanesh/dhanesh/TMYTEK/experiments
2026-03-18 16:13:27.619 - TLKCoreService - INFO : Files        directory: /home/dhanesh/dhanesh/TMYTEK/experiments/files
2026-03-18 16:13:27.620 - TLKCoreService - INFO : Log file     directory: /home/dhanesh/dhanesh/TMYTEK/experiments/tlk_core_log
2026-03-18 16:13:27.620 - TLKCoreService - INFO : Init successfully
2026-03-18 16:13:27.620 - System - INFO : TLKCoreService v2.4.4 | running: True


TMYLogging __init__
Apply logger path to : /home/dhanesh/dhanesh/TMYTEK/experiments/tlk_core_log


2026-03-18 16:13:29.443 - System - INFO : ── Scan results ──────────────────────────
2026-03-18 16:13:29.445 - System - INFO :   SN: UD-BS25020006-24          | IP: 192.168.100.112    | Type: 15
2026-03-18 16:13:29.446 - System - INFO :   SN: UD-BS25020007-24          | IP: 192.168.100.114    | Type: 15
2026-03-18 16:13:29.448 - System - INFO :   SN: D2426E012-28              | IP: 172.25.70.19       | Type: 9
2026-03-18 16:13:29.449 - System - INFO :   SN: D2426E015-28              | IP: 172.25.70.169      | Type: 9


In [13]:
# CELL 3 — Init all 4 devices
# ============================================================
def init_device(sn):
    ret = service.initDev(sn)
    if ret.RetCode is not RetCode.OK:
        logger.error("initDev FAILED [%s]: %s" % (sn, ret.RetMsg))
        return False
    logger.info("initDev OK    [%s]" % sn)
    return True

init_device(SN_UC)
init_device(SN_DC)
init_device(SN_TX)
init_device(SN_RX)


# ============================================================

2026-03-18 16:13:29.480 - System - INFO : initDev OK    [UD-BS25020007-24]
2026-03-18 16:13:29.509 - System - INFO : initDev OK    [UD-BS25020006-24]
2026-03-18 16:13:29.581 - System - INFO : initDev OK    [D2426E012-28]
2026-03-18 16:13:29.646 - System - INFO : initDev OK    [D2426E015-28]


True

In [14]:
# CELL 4 — Configure UP-CONVERTER (UD-BS25020007-24)
# Sets LO so that: RF_out = LO + IF
# ============================================================
def configure_upconverter(LO=24e6, RF=28e6, IF=4e6, BW=1e5):
    logger.info("── Up-Converter ──────────────────────────")
    logger.info("PLO lock : %r" % service.getUDState(SN_UC, UDState.PLO_LOCK).RetData)

    # Enable CH1
    service.setUDState(SN_UC, 1, UDState.CH1)

    # Harmonic check
    hc = service.getHarmonic(SN_UC, LO, IF, BW)
    logger.info("Harmonic check: %r" % hc.RetData)

    # Set freq
    ret = service.setUDFreq(SN_UC, LO, RF, IF, BW)
    logger.info("setUDFreq: %s" % ret.RetCode)

    time.sleep(0.5)
    logger.info("PLO lock after: %r" % service.getUDState(SN_UC, UDState.PLO_LOCK).RetData)
    logger.info("Freq readback : %s" % service.getUDFreq(SN_UC))

# LO=24GHz, RF=28GHz, IF=4GHz, BW=100MHz
configure_upconverter(LO=24e6, RF=28e6, IF=4e6, BW=1e5)


# ============================================================

2026-03-18 16:13:29.659 - System - INFO : ── Up-Converter ──────────────────────────
2026-03-18 16:13:29.665 - System - INFO : PLO lock : 1
2026-03-18 16:13:29.671 - System - INFO : Harmonic check: False
2026-03-18 16:13:29.718 - System - INFO : setUDFreq: OK
2026-03-18 16:13:30.222 - System - INFO : PLO lock after: 1
2026-03-18 16:13:30.227 - System - INFO : Freq readback : {'UDFreq': 24000000, 'RFFreq': 0, 'IFFreq': 0}


In [15]:
# CELL 5 — Configure DOWN-CONVERTER (UD-BS25020006-24)
# Sets LO so that: IF_out = RF_in - LO
# ============================================================
def configure_downconverter(LO=24e6, RF=28e6, IF=4e6, BW=1e5):
    logger.info("── Down-Converter ────────────────────────")
    logger.info("PLO lock : %r" % service.getUDState(SN_DC, UDState.PLO_LOCK).RetData)

    # Enable CH1
    service.setUDState(SN_DC, 1, UDState.CH1)

    # Harmonic check
    hc = service.getHarmonic(SN_DC, LO, IF, BW)
    logger.info("Harmonic check: %r" % hc.RetData)

    # Set freq
    ret = service.setUDFreq(SN_DC, LO, RF, IF, BW)
    logger.info("setUDFreq: %s" % ret.RetCode)

    time.sleep(0.5)
    logger.info("PLO lock after: %r" % service.getUDState(SN_DC, UDState.PLO_LOCK).RetData)
    logger.info("Freq readback : %s" % service.getUDFreq(SN_DC))

# Same freq plan — both converters must share LO/RF/IF
configure_downconverter(LO=24e6, RF=28e6, IF=4e6, BW=1e5)


# ============================================================

2026-03-18 16:13:30.237 - System - INFO : ── Down-Converter ────────────────────────
2026-03-18 16:13:30.242 - System - INFO : PLO lock : 1
2026-03-18 16:13:30.247 - System - INFO : Harmonic check: False
2026-03-18 16:13:30.293 - System - INFO : setUDFreq: OK
2026-03-18 16:13:30.798 - System - INFO : PLO lock after: 1
2026-03-18 16:13:30.802 - System - INFO : Freq readback : {'UDFreq': 24000000, 'RFFreq': 0, 'IFFreq': 0}


In [19]:
# CELL 6 — Configure TX BEAMFORMER (D2426E012-28)
# ============================================================
def configure_tx_beamformer(target_freq=28.0, theta=0, phi=0):
    logger.info("── TX Beamformer ─────────────────────────")
    sn = SN_TX

    # FW/HW info
    logger.info("FW: %s | HW: %s" % (service.queryFWVer(sn).RetData,
                                      service.queryHWVer(sn).RetData))

    # Set TX mode
    ret = service.setRFMode(sn, RFMode.TX)
    logger.info("setRFMode TX: %s" % ret.RetCode)

    # Check available calibration frequencies
    freq_list = service.getFrequencyList(sn).RetData
    logger.info("Available freqs: %s" % freq_list)
    if target_freq not in freq_list:
        logger.error("%.1f GHz not in calibration table!" % target_freq)
        return False

    # Set operating frequency (loads calibration table)
    ret = service.setOperatingFreq(sn, target_freq)
    logger.info("setOperatingFreq %.1fGHz: %s" % (target_freq, ret.RetCode))

    # Get dynamic range
    rng = service.getDR(sn, RFMode.TX).RetData
    gain_max = rng[1]
    logger.info("DR range: %s | gain_max: %.1f" % (rng, gain_max))

    # Select AAKit
    aakit_selected = False
    for aakit in service.getAAKitList(sn).RetData:
        if '8x8' in aakit:
            service.selectAAKit(sn, aakit)
            aakit_selected = True
            logger.info("AAKit selected: %s" % aakit)
            break
    if not aakit_selected:
        logger.warning("No 8x8 AAKit found — PhiA mode")

    # Set beam angle
    if aakit_selected:
        ret = service.setBeamAngle(sn, gain_max, theta, phi)
        logger.info("setBeamAngle (theta=%d, phi=%d): %s" % (theta, phi, ret))
    else:
        logger.error("Cannot steer beam without AAKit")
        return False

    return True

configure_tx_beamformer(target_freq=28.0, theta=0, phi=0)


# ============================================================

2026-03-18 16:13:47.806 - System - INFO : ── TX Beamformer ─────────────────────────
2026-03-18 16:13:47.844 - System - INFO : FW: v1.2.17 | HW: 2437-28MO100E-073
2026-03-18 16:13:47.852 - System - INFO : setRFMode TX: OK
2026-03-18 16:13:47.854 - System - INFO : Available freqs: [26.5, 27.0, 27.5, 28.0, 28.5, 29.0, 29.5]
2026-03-18 16:13:48.740 - System - INFO : setOperatingFreq 28.0GHz: OK
2026-03-18 16:13:48.740 - System - INFO : DR range: [-4.0, 12.0] | gain_max: 12.0
2026-03-18 16:13:48.741 - System - INFO : AAKit selected: TMYTEK_28ONE_8x8_2251-O2BAK81A-016
2026-03-18 16:13:48.748 - System - INFO : setBeamAngle (theta=0, phi=0): [RetType] OK


True

In [20]:
# CELL 7 — Configure RX BEAMFORMER (D2426E015-28)
# ============================================================
def configure_rx_beamformer(target_freq=28.0, theta=0, phi=0):
    logger.info("── RX Beamformer ─────────────────────────")
    sn = SN_RX

    # FW/HW info
    logger.info("FW: %s | HW: %s" % (service.queryFWVer(sn).RetData,
                                      service.queryHWVer(sn).RetData))

    # Set RX mode
    ret = service.setRFMode(sn, RFMode.RX)
    logger.info("setRFMode RX: %s" % ret.RetCode)

    # Check available calibration frequencies
    freq_list = service.getFrequencyList(sn).RetData
    logger.info("Available freqs: %s" % freq_list)
    if target_freq not in freq_list:
        logger.error("%.1f GHz not in calibration table!" % target_freq)
        return False

    # Set operating frequency (loads calibration table)
    ret = service.setOperatingFreq(sn, target_freq)
    logger.info("setOperatingFreq %.1fGHz: %s" % (target_freq, ret.RetCode))

    # Get dynamic range
    rng = service.getDR(sn, RFMode.RX).RetData
    gain_max = rng[1]
    logger.info("DR range: %s | gain_max: %.1f" % (rng, gain_max))

    # Select AAKit
    aakit_selected = False
    for aakit in service.getAAKitList(sn).RetData:
        if '8x8' in aakit:
            service.selectAAKit(sn, aakit)
            aakit_selected = True
            logger.info("AAKit selected: %s" % aakit)
            break
    if not aakit_selected:
        logger.warning("No 8x8 AAKit found — PhiA mode")

    # Set beam angle (must match TX pointing direction)
    if aakit_selected:
        ret = service.setBeamAngle(sn, gain_max, theta, phi)
        logger.info("setBeamAngle (theta=%d, phi=%d): %s" % (theta, phi, ret))
    else:
        logger.error("Cannot steer beam without AAKit")
        return False

    return True

configure_rx_beamformer(target_freq=28.0, theta=0, phi=0)


# ============================================================


2026-03-18 16:13:50.305 - System - INFO : ── RX Beamformer ─────────────────────────
2026-03-18 16:13:50.331 - System - INFO : FW: v1.2.17 | HW: 2437-28MO100E-015
2026-03-18 16:13:50.334 - System - INFO : setRFMode RX: OK
2026-03-18 16:13:50.334 - System - INFO : Available freqs: [26.5, 27.0, 27.5, 28.0, 28.5, 29.0, 29.5]
2026-03-18 16:13:51.342 - System - INFO : setOperatingFreq 28.0GHz: OK
2026-03-18 16:13:51.343 - System - INFO : DR range: [-16.0, 3.5] | gain_max: 3.5
2026-03-18 16:13:51.343 - System - INFO : AAKit selected: TMYTEK_28ONE_8x8_2251-O2BAK81A-016
2026-03-18 16:13:51.355 - System - INFO : setBeamAngle (theta=0, phi=0): [RetType] OK


True

In [18]:
# CELL 8 — Steer both beams together (convenience function)
# Call this any time you want to change pointing direction
# ============================================================
def steer_beams(theta, phi, gain_tx=None, gain_rx=None):
    """
    Point TX and RX beamformers to the same angle.
    Args:
        theta: elevation angle (degrees)
        phi  : azimuth angle (degrees)
        gain_tx: override TX gain (uses max DR if None)
        gain_rx: override RX gain (uses max DR if None)
    """
    logger.info("── Steer beams → theta=%d, phi=%d ────────" % (theta, phi))

    # TX
    rng_tx = service.getDR(SN_TX, RFMode.TX).RetData
    g_tx = gain_tx if gain_tx is not None else rng_tx[1]
    ret = service.setBeamAngle(SN_TX, g_tx, theta, phi)
    logger.info("TX beam: %s" % ret)

    # RX
    rng_rx = service.getDR(SN_RX, RFMode.RX).RetData
    g_rx = gain_rx if gain_rx is not None else rng_rx[1]
    ret = service.setBeamAngle(SN_RX, g_rx, theta, phi)
    logger.info("RX beam: %s" % ret)

# Example — point both beams to 30° elevation, 0° azimuth
steer_beams(theta=10, phi=256)


# ============================================================

2026-03-18 16:13:33. 65 - System - INFO : ── Steer beams → theta=10, phi=256 ────────
2026-03-18 16:13:33. 72 - System - INFO : TX beam: [RetType] OK
2026-03-18 16:13:33. 81 - System - INFO : RX beam: [RetType] OK


In [10]:
# CELL 9 — Cleanup
# ============================================================
def cleanup():
    for sn in [SN_UC, SN_DC, SN_TX, SN_RX]:
        service.DeInitDev(sn)
        logger.info("DeInit: %s" % sn)

cleanup()

2026-03-18 16:13:19.297 - System - INFO : DeInit: UD-BS25020007-24
2026-03-18 16:13:19.300 - System - INFO : DeInit: UD-BS25020006-24
2026-03-18 16:13:19.301 - System - INFO : DeInit: D2426E012-28
2026-03-18 16:13:19.302 - System - INFO : DeInit: D2426E015-28
